# Different Phases
Let's define different phases in our experiment
- Phase 1: Classify IoT vs non-IoT
- Phase 2: Two directions
    - Only classify device type
    - Classify device type and vendor
- Phase 3: Classify events

In [1]:
import os 
import sys
import pandas as pd 
from typing import Dict, List
from pathlib import Path
from scipy.stats import skew, kurtosis
from package import training_and_evaluation

In [2]:
DATASET_FOLDER = "../../../dataset/processed-data"
DEVICE_MAPPING_FOLDER = DATASET_FOLDER + "/device-mapping"
INPUT_FOLDER = DATASET_FOLDER + "/input-data"
DATASET1 = "dataset1"
DATASET2 = "dataset2"

In [3]:
MAPPING_LIST_1 = f"{DEVICE_MAPPING_FOLDER}/{DATASET1}/list_of_devices_processed.csv"
MAPPING_LIST_2 = f"{DEVICE_MAPPING_FOLDER}/{DATASET2}/list_of_devices_processed.csv"
INPUT_1 = f"{INPUT_FOLDER}/{DATASET1}"
INPUT_2 = f"{INPUT_FOLDER}/{DATASET2}"

In [4]:
pd.set_option('display.max_columns', None)
pd.set_option("display.float_format", "{:.6f}".format)

# Dataset 1 - Identify IoT devices and events dataset
## Extract raw data and device mapping list

In [5]:
csv_files_1 = [f"{INPUT_1}/{f.name}" for f in sorted(Path(INPUT_1).iterdir()) if f.is_file() and f.suffix == ".csv"]
for file in csv_files_1:
    print(file)

../../../dataset/processed-data/input-data/dataset1/16-09-23.csv
../../../dataset/processed-data/input-data/dataset1/16-09-24.csv
../../../dataset/processed-data/input-data/dataset1/16-09-25.csv
../../../dataset/processed-data/input-data/dataset1/16-09-26.csv
../../../dataset/processed-data/input-data/dataset1/16-09-27.csv
../../../dataset/processed-data/input-data/dataset1/16-09-28.csv
../../../dataset/processed-data/input-data/dataset1/16-09-29.csv
../../../dataset/processed-data/input-data/dataset1/16-09-30.csv
../../../dataset/processed-data/input-data/dataset1/16-10-01.csv
../../../dataset/processed-data/input-data/dataset1/16-10-02.csv
../../../dataset/processed-data/input-data/dataset1/16-10-03.csv
../../../dataset/processed-data/input-data/dataset1/16-10-04.csv
../../../dataset/processed-data/input-data/dataset1/16-10-05.csv
../../../dataset/processed-data/input-data/dataset1/16-10-06.csv
../../../dataset/processed-data/input-data/dataset1/16-10-07.csv
../../../dataset/processe

In [6]:
device_mapping_1 = pd.read_csv(MAPPING_LIST_1)
device_dict_1 = (
    device_mapping_1
    .set_index("mac_address")
    .drop(columns=["mac_address"], errors="ignore")
    .to_dict(orient="index")
)
device_dict_1

{'44:65:0d:56:cc:d3': {'device_name': 'Amazon Echo',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'smart speaker',
  'vendor': 'Amazon'},
 '70:ee:50:18:34:43': {'device_name': 'Netatmo Welcome',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'Netatmo'},
 'f4:f2:6d:93:51:f1': {'device_name': 'TP-Link Day Night Cloud camera',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'TP-Link'},
 '00:16:6c:ab:6b:88': {'device_name': 'Samsung SmartCam',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'Samsung'},
 '30:8c:fb:2f:e4:b2': {'device_name': 'Dropcam',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'Dropcam'},
 '00:62:6e:51:27:2e': {'device_name': 'Insteon Camera',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'Insteon'},
 'e8:ab:fa:19:de:4f': {'device_name': 'Insteon Camera',
  'ip_address': nan,
  'is_io

In [7]:
df = pd.read_csv(csv_files_1[0])

/var/folders/rv/241jb89d0rn0z37gwxcvxhxc0000gn/T/ipykernel_13000/1465641618.py:1: DtypeWarning: Columns (0: eth.trailer, 1: ip.version, 2: ip.hdr_len, 3: ip.len, 4: ip.frag_offset, 5: ip.ttl, 6: ip.proto, 7: ip.checksum.status, 8: igmp.type) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_files_1[0])


In [8]:
df.head(10)

,frame.number,frame.time_epoch,frame.time_relative,frame.time_delta,frame.cap_len,frame.len,frame.interface_id,frame.encap_type,frame.protocols,eth.src,eth.dst,eth.src.oui,eth.dst.oui,eth.type,eth.len,eth.padding,eth.trailer,eth.fcs,eth.fcs.status,vlan.id,vlan.priority,vlan.etype,arp.opcode,arp.src.hw_mac,arp.src.proto_ipv4,arp.dst.hw_mac,arp.dst.proto_ipv4,ip.version,ip.hdr_len,ip.dsfield,ip.len,ip.id,ip.flags,ip.flags.df,ip.flags.mf,ip.frag_offset,ip.ttl,ip.proto,ip.checksum,ip.checksum.status,ip.src,ip.dst,ipv6.version,ipv6.tclass,ipv6.flow,ipv6.plen,ipv6.nxt,ipv6.hlim,ipv6.src,ipv6.dst,icmp.type,icmp.code,icmp.checksum,icmpv6.type,icmpv6.code,igmp.type,igmp.version
0,1,1474552802.257569,0.000000,NaN,70,70,0,1,eth:ethertype:ip:tcp:data,18:b7:9e:02:20:44,14:cc:20:51:33:ea,1619870,1362976,0x0800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,20,0x00,56,0x8492,0x02,True,False,0,64,6,0x9f55,2,192.168.1.120,46.105.38.79,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1474552802.257692,0.000123,0.000123,66,66,0,1,eth:ethertype:ip:tcp,18:b7:9e:02:20:44,14:cc:20:51:33:ea,1619870,1362976,0x0800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,20,0x00,52,0x8493,0x02,True,False,0,64,6,0x9f58,2,192.168.1.120,46.105.38.79,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1474552802.323085,0.065516,0.065393,66,66,0,1,eth:ethertype:ip:tcp,14:cc:20:51:33:ea,30:8c:fb:2f:e4:b2,1362976,3181819,0x0800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,20,0x00,52,0xd580,0x02,True,False,0,224,6,0xdd39,2,52.87.241.159,192.168.1.106,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1474552802.383174,0.125605,0.060089,60,60,0,1,eth:ethertype:ip:tcp,d0:52:a8:00:67:5e,14:cc:20:51:33:ea,13652648,1362976,0x0800,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,20,0x00,41,0x69b0,0x00,False,False,0,100,6,0xce37,2,192.168.1.196,52.1.40.122,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1474552802.396326,0.138757,0.013152,98,98,0,1,eth:ethertype:ip:icmp:data,14:cc:20:51:33:ea,70:ee:50:18:34:43,1362976,7401040,0x0800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,20,0x00,84,0x5956,0x00,False,False,0,45,1,0xe671,2,195.154.199.173,192.168.1.241,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0xb990,NaN,NaN,NaN,NaN
5,6,1474552802.447911,0.190342,0.051585,88,88,0,1,eth:ethertype:ip:udp:dns,70:ee:50:18:34:43,14:cc:20:51:33:ea,7401040,1362976,0x0800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,20,0x00,74,0x6677,0x02,True,False,0,64,17,0x4fe9,2,192.168.1.241,192.168.1.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,1474552802.555796,0.298227,0.107885,68,68,0,1,eth:ethertype:ip:tcp:data,14:cc:20:51:33:ea,18:b7:9e:02:20:44,1362976,1619870,0x0800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,20,0x00,54,0x6d75,0x02,True,False,0,48,6,0xc674,2,46.105.38.79,192.168.1.120,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,8,1474552802.587523,0.329954,0.031727,54,54,0,1,eth:ethertype:ip:tcp,14:cc:20:51:33:ea,d0:52:a8:00:67:5e,1362976,13652648,0x0800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,20,0x00,40,0xc44c,0x02,True,False,0,32,6,0x779c,2,52.1.40.122,192.168.1.196,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9,1474552802.634732,0.377163,0.047209,42,42,0,1,eth:ethertype:arp,ec:1a:59:79:f4:89,14:cc:20:51:33:ea,15473241,1362976,0x0806,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,ec:1a:59:79:f4:89,192.168.1.223,00:00:00:00:00:00,192.168.1.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,10,1474552802.634774,0.377205,0.000042,42,42,0,1,eth:ethertype:arp,14:cc:20:51:33:ea,ec:1a:59:79:f4:89,1362976,15473241,0x0806,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000000,14:cc:20:51:33:ea,192.168.1.1,ec:1a:59:79:f4:89,192.168.1.223,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


- There are multiple empty entries for IP address, thus I will be using MAC address to label the traffic
## Feature Set 1 - Statistical Distribution Features

In [9]:
FIXED_WINDOW = 2 # 2 seconds

In [10]:
from collections.abc import Iterable
import pandas as pd


def combine_and_extract(file_paths: Iterable[str]) -> pd.DataFrame:
    columns = ["frame.time_epoch", "frame.len", "eth.src", "eth.dst"]
    all_features = []

    for file_path in file_paths:
        df = pd.read_csv(
            file_path,
            usecols=columns,
            dtype={
                "frame.time_epoch": "float64",
                "frame.len": "int32",
                "eth.src": "category",
                "eth.dst": "category",
            },
        )

        start = df["frame.time_epoch"].min()

        df["window"] = (
            (df["frame.time_epoch"] - start) // FIXED_WINDOW
        ).astype("int32")

        grouped = df.groupby(
            ["window", "eth.src"],
            sort=False,
            observed=True,
        )["frame.len"]

        basic_features = grouped.agg(
            packet_count="size",
            total_bytes="sum",
            frame_length_mean="mean",
        )

        basic_features["frame_length_std"] = grouped.std(ddof=0)

        quantiles = (
            grouped
            .quantile([0.25, 0.50, 0.75, 0.95])
            .unstack()
            .rename(
                columns={
                    0.25: "frame_length_q25",
                    0.50: "frame_length_median",
                    0.75: "frame_length_q75",
                    0.95: "frame_length_q95",
                }
            )
        )

        features = (
            basic_features
            .join(quantiles)
            .reset_index()
        )

        all_features.append(features)

    return pd.concat(all_features, ignore_index=True)

In [11]:
def label_features(features: pd.DataFrame, device_mapping: Dict[str, Dict[str, str]]) -> pd.DataFrame:
    # Check if eth.src is in device_mapping, and get label is_iot_device, device_type, and vendor
    features["is_iot_device"] = features["eth.src"].apply(
        lambda x: device_mapping.get(x, {}).get("is_iot_device", 0)
    )
    features["device_type"] = features["eth.src"].apply(
        lambda x: device_mapping.get(x, {}).get("device_type", "unknown")
    )
    features["vendor"] = features["eth.src"].apply(
        lambda x: device_mapping.get(x, {}).get("vendor", "unknown")
    )
    return features
    

In [12]:
all_data = combine_and_extract(csv_files_1)

In [13]:
labeled_data = label_features(all_data, device_dict_1)

In [14]:
labeled_data.head(10)

,window,eth.src,packet_count,total_bytes,frame_length_mean,frame_length_std,frame_length_q25,frame_length_median,frame_length_q75,frame_length_q95,is_iot_device,device_type,vendor
0,0,18:b7:9e:02:20:44,3,202,67.333333,1.885618,66.000000,66.000000,68.000000,69.600000,1,smart speaker,Invoxia
1,0,14:cc:20:51:33:ea,12,1045,87.083333,53.635278,54.000000,67.000000,98.000000,178.800000,0,unknown,unknown
2,0,d0:52:a8:00:67:5e,3,259,86.333333,37.240957,60.000000,60.000000,99.500000,131.100000,0,unknown,unknown
3,0,70:ee:50:18:34:43,3,284,94.666667,4.714045,93.000000,98.000000,98.000000,98.000000,1,camera,Netatmo
4,0,ec:1a:59:79:f4:89,1,42,42.000000,0.000000,42.000000,42.000000,42.000000,42.000000,1,smart switch,Belkin
5,0,30:8c:fb:2f:e4:b2,2,312,156.000000,0.000000,156.000000,156.000000,156.000000,156.000000,1,camera,Dropcam
6,1,14:cc:20:51:33:ea,6,404,67.333333,16.275407,66.000000,66.000000,66.000000,90.000000,0,unknown,unknown
7,1,00:24:e4:11:18:a8,2,132,66.000000,0.000000,66.000000,66.000000,66.000000,66.000000,1,baby monitor,Withings
8,1,30:8c:fb:2f:e4:b2,2,312,156.000000,0.000000,156.000000,156.000000,156.000000,156.000000,1,camera,Dropcam
9,2,08:21:ef:3b:fc:e3,2,108,54.000000,12.000000,48.000000,54.000000,60.000000,64.800000,0,unknown,unknown


### Phase 1 Experiment

In [15]:
training_features = ["packet_count", "total_bytes", "frame_length_mean", "frame_length_std", "frame_length_q25", "frame_length_median", "frame_length_q75", "frame_length_q95"]
label = "is_iot_device"
X, y = labeled_data[training_features], labeled_data[label]

In [16]:
X.isna().sum()

packet_count           0
total_bytes            0
frame_length_mean      0
frame_length_std       0
frame_length_q25       0
frame_length_median    0
frame_length_q75       0
frame_length_q95       0
dtype: int64

In [17]:
y.value_counts()

is_iot_device
1    2712435
0    1187183
Name: count, dtype: int64

In [18]:
# first_experiment_results = training_and_evaluation(X, y)
# first_experiment_results.to_csv("first_experiment_results.csv", index=False)

### Phase 2 Experiment - Identify Device Types

In [19]:
training_features = ["packet_count", "total_bytes", "frame_length_mean", "frame_length_std", "frame_length_q25", "frame_length_median", "frame_length_q75", "frame_length_q95"]
label = "device_type"
filtered_data = labeled_data[labeled_data["is_iot_device"] == 1]
X, y = filtered_data[training_features], filtered_data[label]

In [20]:
X.isna().sum()

packet_count           0
total_bytes            0
frame_length_mean      0
frame_length_std       0
frame_length_q25       0
frame_length_median    0
frame_length_q75       0
frame_length_q95       0
dtype: int64

In [21]:
y.value_counts()

device_type
camera                    1431035
smart speaker              319566
baby monitor               274314
motion sensor              137029
smart switch               132311
sleep sensor               130399
printer                    106986
smart bulb                  60660
weather station             60624
smart plug                  42040
photo frame                 16587
smart scale                   541
smoke alarm                   320
blood pressure monitor         23
Name: count, dtype: int64

In [22]:
second_experiment_results = training_and_evaluation(X, y)
second_experiment_results.to_csv("second_experiment_results.csv", index=False)

Training and evaluating with:
Features: ['packet_count', 'total_bytes', 'frame_length_mean', 'frame_length_std', 'frame_length_q25', 'frame_length_median', 'frame_length_q75', 'frame_length_q95']
Label: ['baby monitor' 'blood pressure monitor' 'camera' 'motion sensor'
 'photo frame' 'printer' 'sleep sensor' 'smart bulb' 'smart plug'
 'smart scale' 'smart speaker' 'smart switch' 'smoke alarm'
 'weather station']
	Start KStratifiedKFold (k=5) for: model=DecisionTree, sample_method=RandomUnderSampler, scaler=StandardScaler
	Start KStratifiedKFold (k=5) for: model=RandomForest, sample_method=RandomUnderSampler, scaler=StandardScaler
	Start KStratifiedKFold (k=5) for: model=DecisionTree, sample_method=SMOTE, scaler=StandardScaler
	Start KStratifiedKFold (k=5) for: model=RandomForest, sample_method=SMOTE, scaler=StandardScaler


In [23]:
second_experiment_results

,accuracy,precision_class_baby monitor,recall_class_baby monitor,f1_class_baby monitor,precision_class_blood pressure monitor,recall_class_blood pressure monitor,f1_class_blood pressure monitor,precision_class_camera,recall_class_camera,f1_class_camera,precision_class_motion sensor,recall_class_motion sensor,f1_class_motion sensor,precision_class_photo frame,recall_class_photo frame,f1_class_photo frame,precision_class_printer,recall_class_printer,f1_class_printer,precision_class_sleep sensor,recall_class_sleep sensor,f1_class_sleep sensor,precision_class_smart bulb,recall_class_smart bulb,f1_class_smart bulb,precision_class_smart plug,recall_class_smart plug,f1_class_smart plug,precision_class_smart scale,recall_class_smart scale,f1_class_smart scale,precision_class_smart speaker,recall_class_smart speaker,f1_class_smart speaker,precision_class_smart switch,recall_class_smart switch,f1_class_smart switch,precision_class_smoke alarm,recall_class_smoke alarm,f1_class_smoke alarm,precision_class_weather station,recall_class_weather station,f1_class_weather station,model,sample_method,iteration,labels
0,0.590779,0.676295,0.762226,0.716694,0.000174,0.800000,0.000348,0.998122,0.635166,0.776315,0.326076,0.337201,0.331545,0.150226,0.100090,0.120138,0.376894,0.380082,0.378482,0.285273,0.870279,0.429694,0.685550,0.499753,0.578089,0.225532,0.268316,0.245071,0.006190,0.703704,0.012273,0.551038,0.431095,0.483742,0.320385,0.410475,0.359877,0.003188,0.765625,0.006349,0.716478,0.793237,0.752906,DecisionTree,RandomUnderSampler,0,"[baby monitor, blood pressure monitor, camera,..."
1,0.572974,0.688700,0.977288,0.807999,0.001197,0.750000,0.002389,0.966833,0.522391,0.678293,0.575414,0.268664,0.366300,0.051626,0.453586,0.092701,0.387311,0.504136,0.438069,0.308915,0.878144,0.457048,0.380487,0.577316,0.458677,0.207196,0.721218,0.321911,0.005176,0.694444,0.010275,0.787350,0.547682,0.646003,0.326280,0.301251,0.313266,0.002612,0.687500,0.005204,0.573765,0.740619,0.646601,DecisionTree,RandomUnderSampler,1,"[baby monitor, blood pressure monitor, camera,..."
2,0.595247,0.774783,0.804313,0.789272,0.000321,0.750000,0.000641,0.962590,0.572411,0.717911,0.400863,0.345618,0.371196,0.074405,0.110307,0.088867,0.284511,0.906482,0.433091,0.318892,0.869287,0.466611,0.346157,0.605918,0.440602,0.452730,0.641889,0.530966,0.008231,0.669725,0.016262,0.787555,0.553111,0.649835,0.571450,0.281385,0.377089,0.001599,0.718750,0.003191,0.781280,0.609287,0.684647,DecisionTree,RandomUnderSampler,2,"[baby monitor, blood pressure monitor, camera,..."
3,0.569733,0.732761,0.803055,0.766299,0.000420,0.600000,0.000839,0.944632,0.572386,0.712839,0.489150,0.384113,0.430315,0.036813,0.387398,0.067237,0.262388,0.634762,0.371296,0.265578,0.850307,0.404742,0.556888,0.524481,0.540199,0.449844,0.633088,0.525962,0.012237,0.759259,0.024086,0.655575,0.514794,0.576717,0.376344,0.072746,0.121924,0.002480,0.562500,0.004939,0.676932,0.576495,0.622689,DecisionTree,RandomUnderSampler,3,"[baby monitor, blood pressure monitor, camera,..."
4,0.571595,0.793627,0.799008,0.796309,0.000401,0.800000,0.000802,0.975905,0.587005,0.733070,0.598258,0.240604,0.343187,0.030080,0.339463,0.055262,0.298497,0.673833,0.413722,0.756304,0.593443,0.665048,0.127350,0.801682,0.219787,0.300971,0.549358,0.388887,0.019438,0.768519,0.037917,0.590242,0.526012,0.556279,0.468510,0.207467,0.287585,0.002987,0.718750,0.005949,0.897730,0.580619,0.705164,DecisionTree,RandomUnderSampler,4,"[baby monitor, blood pressure monitor, camera,..."
5,0.575769,0.739881,0.673678,0.705229,0.000288,0.800000,0.000575,0.994111,0.641710,0.779952,0.334689,0.345302,0.339913,0.014473,0.330721,0.027733,0.446533,0.379755,0.410445,0.825400,0.692818,0.753320,0.652970,0.505605,0.569915,0.282787,0.257731,0.269678,0.004818,0.648148,0.009565,0.667565,0.410708,0.508544,0.314915,0.401255,0.352881,0.002707,0.765625,0.005396,0.358614,0.798268,0.494900,RandomForest,RandomUnderSampler,0,"[baby monitor, blood pressure monitor, camera,..."
6,0.589505,0.705149,0.977288,0.819